# S13: Integral


In [ ]:
import numpy as np
import sympy as sy
import matplotlib.pyplot as plt
from sympy import Function, diff, Symbol, latex
from sympy.plotting import plot
from IPython.display import Math, display
import ipywidgets as widgets
from ipywidgets import interact

# Lower and Upper Sums

How can we estimate the area under the parabola $y = x^2$ over the interval $[0, 1]$?



In [ ]:
# Define the symbolic variable
x = sy.Symbol('x')

# Define the functions
f = x**2

f = 1 + (x - 1) * (x - 2) * (x - 4)
#f = x

# Define the interval
a = 1
b = 3

In [ ]:
plot(f, (x, a, b), title=f"Plot of $f(x) = {latex(f)}$ on $[{a}, {b}]$");

In [ ]:
# Use sympy to compute the integral of f on [a, b]

Integral = sy.integrate(f, (x, a, b))
Integral, float(Integral)

We can compute upper and lower bounds for the area by defining a partition of the interval in $N$ subintervals:
$$L(f, P) = \sum_{k = 1}^N m_k (x_k - x_{k - 1}), \qquad
U(f, P) = \sum_{k = 1}^N m_k (x_k - x_{k - 1}) $$
where
$$
\begin{cases}
m_k = \text{absolute minimum of } f \text{ in }I_k\\[3mm]
M_k = \text{absolute minimum of } f \text{ in }I_k\\
\end{cases}
$$
We begin by deciding the number $N$ of subintervals in the partition $P$. IN this case we will use equal-size intervals, and we use $N$ to compute their common length `dx` (which therefore equals the diameter of the partition).

In [ ]:
N = 100
dx = (b - a) / N

And now we use a list comprehension to get a list with all the subintervals. This list is the partition $P$. The `x_left` and `x_right` lists define the endpoints of the subintervals and will be handy in the rest of our work.

In [ ]:
x_left = [a + k * dx for k in range(N)]
x_right = [a + (k + 1) * dx for k in range(N)]

P = [sy.Interval(x_left[k], x_right[k]) for k in range(N)]

We can examine the first and last subintervals of the partition:

In [ ]:
P[:5], P[-5:]


### Computing $m_k$ and $M_k$

Note that in the case of the parabola the minimum occurs at the leftmost point of $I_k$ while the maximun occurs at the rightmost point. For other functions, determining $m_k$ and $m_k$ can be potentially very difficult (if at all possible). We can however try to use the `maximum` and `minimum` functions provided by sympy. For example if you ask for the extreme values of $x^2$ in $[0.1, 0.3]$ you get:

In [ ]:
#from sympy import Interval, maximum, minimum
sy.minimum(f, x, sy.Interval(0.1, 0.3)), sy.maximum(f, x, sy.Interval(0.1, 0.3))

So we can obtain lists of the $m_k$ and $M_k$ as follows.

In [ ]:
use_sympy_extremes = True
if use_sympy_extremes:
  mk = [sy.minimum(f, x, Ik) for Ik in P]
  Mk = [sy.maximum(f, x, Ik) for Ik in P]

In general, if $N$ is large this is a very slow procedure. If you happen to know that the function is monotonic (increasing or decreasing) you can speed things up a lot by simply evaluating the function at the corresponding endpoint of $I_k$.

In [ ]:
f_increasing = False
if f_increasing:
  m_k = [f.subs(x, xi) for xi in x_left]
  M_k = [f.subs(x, xi) for xi in x_right]

f_decreasing = False
if f_decreasing:
  m_k = [f.subs(x, xi) for xi in x_right]
  M_k = [f.subs(x, xi) for xi in x_left]

Whatever the procedure we use, we are now ready to get the lower and upper sums:

In [ ]:
lower_sum = sum(mk) * dx
upper_sum = sum(Mk) * dx

print(f"Lower sum: {float(lower_sum)}")
print(f"Upper sum: {float(upper_sum)}")

# Interactive Plot of the Lower and Upper Sums


To visualize this and make it a dynamic plot that depends on the number of intervals we are going to use the `ipywidgets`library. The simplest way is to create  a functions that depends on `N`.

In [ ]:
def plot_upper_lower_sums(N_val):

    # The first part of the function repeats the m_k and M_k computation
    N = N_val
    dx = (b - a) / N_val

    x_left = [a + k * dx for k in range(N_val)]
    x_right = [a + (k + 1) * dx for k in range(N_val)]

    m_k = [f.subs(x, xi) for xi in x_left]
    lower_sum = sum(m_k) * dx

    M_k = [f.subs(x, xi) for xi in x_right]
    upper_sum = sum(M_k) * dx




    # We create many x_vals and y_vals to plot the graph of the function
    x_vals = np.linspace(a, b, 500)
    y_vals = [f.subs(x, val) for val in x_vals]

    # Plot the function
    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y_vals, color='blue', label='$f(x)$')

    # Plot the lower and upper sums
    for i in range(N):
        xi = x_left[i]

        yi = m_k[i]
        plt.bar(xi, yi, width=dx, align='edge', alpha=0.3, color='green', edgecolor='green', label='Lower Sum' if i == 0 else "")

        yi = M_k[i]
        plt.bar(xi, yi, width=dx, align='edge', alpha=0.3, color='red', edgecolor='red', label='Upper Sum' if i == 0 else "")

    plt.title(f'Left and right Riemann sums for $f(x)$ on $[{a}, {b}]$ with N={N}\n Only equals upper and lower for monotonic functions!')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Left Riemann sum with N={N}: {lower_sum}")
    print(f"Right Riemann sum with N={N}: {upper_sum}")


Now that the function is created we can use `interact` fro ipywidgets to make a dynamic plot with a slider to select the value of $N$:

In [ ]:

interact(plot_upper_lower_sums, N_val=widgets.IntSlider(min=1, max=100, step=1, value=10, description='N'));

# Riemann sums

Now we are going to compute a Riemann sum by selecting a random point inside each of the intervals $I_k$ of the partition $P$.

In Python when we want to use random variables we often turn to NumPy. The `random` submodule of NumPy provides many functions that deal with randomization and probability distributions. For example, to pick a random point $xrnd_k$ inside the interval $I_k$ we can use the `np.random.uniform` function as follows.

In [ ]:
xrnd_k = [np.random.uniform(Ik.start, Ik.end) for Ik in P]

Let us show the first five random points we have selected (rounded to 3 digits):

In [ ]:
print(f"Random points from each interval (first 5): {[f'{u:.3f}' for u in xrnd_k[:5]]}")

Computing the Riemann sum is now straightforward. We compute the value of $f$ at each $x_k$

In [ ]:
f_at_xrnd_k = [f.subs(x, u) for u in xrnd_k]

Riemann_sum = sum(f_at_xrnd_k) * dx
print(f"Riemann sum: {Riemann_sum}")

In [ ]:
def plot_Riemann_sums(N_val):

    # The first part of the function repeats the m_k and M_k computation
    N = N_val
    dx = (b - a) / N

    x_left = [a + k * dx for k in range(N)]
    x_right = [a + (k + 1) * dx for k in range(N)]
    P = [sy.Interval(x_left[k], x_right[k]) for k in range(N)]

    xrnd_k = [np.random.uniform(Ik.start, Ik.end) for Ik in P]
    f_at_xrnd_k = [f.subs(x, u) for u in xrnd_k]
    Riemann_sum = sum(f_at_xrnd_k) * dx



    # We create many x_vals and y_vals to plot the graph of the function
    x_vals = np.linspace(a, b, 500)
    y_vals = [f.subs(x, val) for val in x_vals]

    # Plot the function
    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y_vals, color='blue', label='$f(x)$')

    # Plot the lower and upper sums
    for i in range(N):
        xi = x_left[i]

        yi = f_at_xrnd_k[i]
        plt.bar(xi, yi, width=dx, align='edge', alpha=0.3, color='green', edgecolor='green', label='Riemann Sum' if i == 0 else "")


    plt.title(f'Riemann Sums for $f(x)$ on $[{a}, {b}]$ with N={N}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Riemann sum with N={N}: {Riemann_sum}")



In [ ]:
interact(plot_Riemann_sums, N_val=widgets.IntSlider(min=1, max=100, step=1, value=10, description='N'));

# Trapezoid's Rule

Computing the Trapezoid's Rule approximation is very easy from our previous work:

In [ ]:
trapezoids = [dx * (f.subs(x, x_left[i]) + f.subs(x, x_right[i])) / 2   for i in range(N)]
trapezoid_sum = sum(trapezoids)
print(f"Trapezoid sum: {trapezoid_sum} for N = {N}")

And the interactive plot is also very similar:

In [ ]:
def plot_trapezoidal_sum(N_val):

    # The first part of the function repeats the m_k and M_k computation
    N = N_val
    dx = (b - a) / N

    x_left = [a + k * dx for k in range(N)]
    x_right = [a + (k + 1) * dx for k in range(N)]
    f_at_left = [f.subs(x, x_left[i]) for i in range(N)]
    f_at_right = [f.subs(x, x_right[i]) for i in range(N)]
    trapezoids = [dx * (f_at_left[i] + f_at_right[i]) / 2   for i in range(N)]
    trapezoid_sum = sum(trapezoids)


    # We create many x_vals and y_vals to plot the graph of the function
    x_vals = np.linspace(a, b, 500)
    y_vals = [f.subs(x, val) for val in x_vals]

    # Plot the function
    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y_vals, color='blue', label='$f(x)$')

    # Plot the trapezoids
    for i in range(N):
        x_trap = [x_left[i], x_right[i], x_right[i], x_left[i]]
        y_trap = [0, 0, f_at_right[i], f_at_left[i]]
        plt.fill(x_trap, y_trap, color='purple', alpha=0.3, edgecolor='purple', label='Trapezoidal Sum' if i == 0 else "")
        plt.plot([x_left[i], x_right[i]], [f_at_left[i], f_at_right[i]], color='purple', linestyle='--', linewidth=1)




    plt.title(f'Trapezoidal sums for $f(x)$ on $[{a}, {b}]$ with N={N}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Trapezoidal sum with N={N}: {trapezoid_sum}")


In [ ]:
interact(plot_trapezoidal_sum, N_val=widgets.IntSlider(min=1, max=100, step=1, value=10, description='N'));